# 3DINO-ViT Transfer Experiment — 3D self-supervised foundation model → Glaucoma

Apply **[3DINO-ViT](https://github.com/AICONSlab/3DINO)** (npj Digital Medicine 2025, AICONSlab):
a ViT-Large self-supervised **3D medical imaging foundation model** (~307M params, pretrained on
~100K multi-organ 3D scans) to glaucoma detection on **Harvard-GF OCT** (streamed from HF).

> License: 3DINO code + weights are **CC BY-NC-ND 4.0** — academic/research use only, no commercial use.

**Method (this notebook):**
1. Load pretrained 3DINO-ViT from Hugging Face (gated — accept terms + `HF_TOKEN` secret).
2. Resize OCT to **112³** (3DINO's native size, patch 16 → 7³ tokens).
3. **Linear probe** — freeze backbone, extract the 1024-d CLS embedding per volume, fit
   logistic regression → quick transfer baseline (compares with our UNet-encoder sweep).
4. **Head finetune** — freeze backbone, train a linear head with augmentation + AMP + cosine +
   early stop.
5. **Full finetune** — unfreeze the backbone (lr 1e-5, warmup+cosine, layer-wise decay option,
   resume-safe checkpoint) — the main lever to beat the frozen 0.78.
6. **Fairness + ROC + calibration** — per-race accuracy (Harvard-GF is balanced Asian/Black/White),
   ROC/AUC, reliability diagram + Brier score.
7. **Attention saliency** — CLS attention-rollout through all 24 ViT blocks, overlaid on slices
   (shortcut-learning / anatomy check).
8. **3D Grad-CAM on FP/FN** — gradient-weighted attention on false positives/negatives to check
   whether the model highlights the RNFL / neuroretinal rim or relies on background artifacts
   (includes a border-mass artifact heuristic).
9. **Label-efficiency curve** — probe accuracy at 1/5/10/50/100% of labeled train data.

Weights gated: https://huggingface.co/AICONSlab/3DINO-ViT (accept terms first), then Colab secrets → `HF_TOKEN`.


In [ ]:
# Light deps only — do NOT `pip install -r requirements.txt` (3DINO pins torch 2.0 / xformers / cuml).
# The model falls back to standard attention when xformers is absent.
!pip -q install omegaconf fvcore iopath torchmetrics
!pip -q install hf-transfer huggingface_hub

# Clone 3DINO — used only for its `dinov2` model code.
!git clone --depth 1 https://github.com/AICONSlab/3DINO.git /content/3DINO 2>/dev/null || (cd /content/3DINO && git pull -q)
print("[3dino] repo ready at /content/3DINO")


In [ ]:
# HF token (Colab Secrets -> HF_TOKEN -> hf_xxx) + Drive for saving results
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")


In [ ]:
# ====== keep the Colab session alive during long runs ======
# Re-clicks Colab's "connect" button every 60s so an idle browser tab does not
# kill the runtime. Keep this tab open and unfocused is fine — but don't close it.
from google.colab import output

JS = """
setInterval(function(){
  const btn = document.querySelector("colab-connect-button");
  if (btn) btn.click();
}, 60000);
"""
try:
    output.eval_js(JS)
    print("[keepalive] armed — runtime will auto-reconnect while this tab stays open.")
except Exception as e:
    print("[keepalive] not available:", e)


In [ ]:
import sys, os, io, json, time, zipfile
from pathlib import Path
sys.path.insert(0, "/content/3DINO")
from dinov2.configs import load_and_merge_config_3d
from dinov2.eval.setup import build_model_for_eval

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True
_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if _bf16 else torch.float16
SEED = 42
RESOLUTION = 112            # 3DINO-ViT native size (patch 16 -> 7^3 tokens)
FEAT_BS = 16                # feature-extraction batch (frozen backbone)


def preprocess_volume(x):
    """(B,1,D,H,W) uint8 -> [-1,1] per-volume via 0.05%/99.95% percentile clip (3DINO style)."""
    x = x.float()
    b = x.shape[0]
    xf = x.reshape(b, 1, -1)
    lo = torch.quantile(xf, 0.0005, dim=2, keepdim=True).unsqueeze(-1).unsqueeze(-1)
    hi = torch.quantile(xf, 0.9995, dim=2, keepdim=True).unsqueeze(-1).unsqueeze(-1)
    x = (x - lo) / (hi - lo + 1e-6)
    return torch.clip(x * 2 - 1, -1, 1)


In [ ]:
# 3DINO-ViT pretrained weights (GATED repo). Accept terms at
# https://huggingface.co/AICONSlab/3DINO-ViT and set HF_TOKEN in Colab secrets first.
from huggingface_hub import hf_hub_download
import os

try:
    WEIGHTS = hf_hub_download(repo_id="AICONSlab/3DINO-ViT", filename="3dino_vit_weights.pth")
except Exception as e:
    print("[3dino] ERROR downloading gated weights:", e)
    print("  1) Open https://huggingface.co/AICONSlab/3DINO-ViT and click 'Agree and access'")
    print("  2) In Colab: key icon (Secrets) -> add HF_TOKEN = hf_xxx")
    print("  3) Re-run the mount cell, then this cell")
    raise
print("[3dino] weights:", WEIGHTS)


In [ ]:
# ================== 200³ data, streamed from Hugging Face ==================
# harvardairobotics/Harvard-GF  (3,300 scans; 2100 / 300 / 900 train/val/test)
HF_REPO  = "harvardairobotics/Harvard-GF"
ZIP_FILE = "Dataset/dataset.zip"       # per-scan .npz with key 'oct_bscans' (200³ uint8)
CSV_FILE = "ReadMe/data_summary.csv"   # columns: filename,glaucoma(yes/no),use(training/validation/test)
DATA_DIR = "/content/glaucoma_hf_200"  # consolidated .npy arrays land here (cached on disk)
SPLITS   = ("Training", "Validation", "Test")

RESOLUTION  = 112                      # store arrays at this size (resize 200->R if R != 200);
                                       # pick a multiple of 32 for SwinUNETR (e.g. 128/192)
BATCH_SIZE   = 2                       # 200³ -> tiny per-step batch
GRAD_ACCUM   = 8                       # effective batch = 2 * 8 = 16
CACHE_IN_RAM = False                   # 200³ = ~26 GB total -> mmap, never hold in RAM
NUM_WORKERS  = max(2, os.cpu_count() or 2)

SPLIT_ALIAS = {"training": "Training", "validation": "Validation", "valid": "Validation",
               "test": "Test", "testing": "Test"}


def download_hf(filename):
    from huggingface_hub import hf_hub_download
    print(f"[data] downloading {HF_REPO}/{filename} ...", flush=True)
    return hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset")


def build_200_data():
    """Stream Harvard-GF -> per-split 200³ .npy arrays (once, low RAM, disk cached)."""
    if all(os.path.isfile(os.path.join(DATA_DIR, f"{s}_volumes.npy")) for s in SPLITS):
        print(f"[data] already built at {DATA_DIR}")
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    csv_path = download_hf(CSV_FILE)
    zip_path = download_hf(ZIP_FILE)

    import csv
    meta = {}
    with open(csv_path, newline="") as fh:
        for r in csv.DictReader(fh):
            split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
            if split is None:
                continue
            gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
            meta[Path(r["filename"]).stem] = (split, gl)
    print(f"[data] {len(meta)} labeled samples from CSV")

    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if n.endswith(".npz")]
        counts = {s: 0 for s in SPLITS}
        for n in names:
            m = meta.get(Path(n).stem)
            if m:
                counts[m[0]] += 1
    print("[data] zip-matched counts:", counts)

    vols, labels = {}, {}
    for s in SPLITS:
        vp = os.path.join(DATA_DIR, f"{s}_volumes.npy")
        vols[s] = np.lib.format.open_memmap(vp, mode="w+", dtype=np.uint8,
                                            shape=(counts[s], 1, RESOLUTION, RESOLUTION, RESOLUTION))
        labels[s] = np.zeros(counts[s], dtype=np.int64)

    filled = {s: 0 for s in SPLITS}
    with zipfile.ZipFile(zip_path) as zf:
        for n in names:
            m = meta.get(Path(n).stem)
            if not m:
                continue
            split, label = m
            raw = np.load(io.BytesIO(zf.read(n)))["oct_bscans"]      # (200,200,200) uint8
            if RESOLUTION != 200:
                t = torch.from_numpy(raw).float().div_(255.0).unsqueeze(0).unsqueeze(0)
                t = torch.nn.functional.interpolate(
                    t, size=(RESOLUTION,) * 3, mode="trilinear", align_corners=False)
                raw = (t.squeeze(0, 1).clamp(0, 1) * 255).round().numpy().astype(np.uint8)
            vols[split][filled[split]] = raw[None]                   # -> (1,R,R,R)
            labels[split][filled[split]] = label
            filled[split] += 1
    for s in SPLITS:
        vols[s].flush()
        np.save(os.path.join(DATA_DIR, f"{s}_labels.npy"), labels[s])
        print(f"[data] {s}: {filled[s]} volumes ({(counts[s] * 8 / 1e9):.1f} GB)")
    with open(os.path.join(DATA_DIR, "manifest.json"), "w") as fh:
        json.dump({"source": HF_REPO, "size_name": "200", "store_shape": [1, 200, 200, 200],
                   "splits": {s: {"built_n": filled[s]} for s in SPLITS}}, fh, indent=2)


class OCTMemmapDataset(Dataset):
    """Consolidated {split}_volumes.npy is (N,1,200,200,200) uint8; labels (N,) int64."""

    def __init__(self, data_dir, split, cache_in_ram=False):
        self.labels = np.load(os.path.join(data_dir, f"{split}_labels.npy"))
        vp = os.path.join(data_dir, f"{split}_volumes.npy")
        self.volumes = np.load(vp) if cache_in_ram else np.load(vp, mmap_mode="r")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(np.ascontiguousarray(self.volumes[idx]).copy())  # uint8 (1,200,200,200), writable
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


def build_loaders():
    build_200_data()
    counts = {s: len(np.load(os.path.join(DATA_DIR, f"{s}_labels.npy"))) for s in SPLITS}
    print("[data] counts:", counts)
    workers = 0 if CACHE_IN_RAM else NUM_WORKERS
    kw = dict(batch_size=BATCH_SIZE, num_workers=workers, pin_memory=(device == "cuda"))
    if workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    train_ds = OCTMemmapDataset(DATA_DIR, "Training",   cache_in_ram=CACHE_IN_RAM)
    val_ds   = OCTMemmapDataset(DATA_DIR, "Validation", cache_in_ram=CACHE_IN_RAM)
    test_ds  = OCTMemmapDataset(DATA_DIR, "Test",       cache_in_ram=CACHE_IN_RAM)
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders()


In [ ]:
# ====== load pretrained 3DINO-ViT and freeze it ======
cfg = load_and_merge_config_3d("train/vit3d_highres")
model = build_model_for_eval(cfg, WEIGHTS)          # DinoVisionTransformer3d, moved to cuda
for p in model.parameters():
    p.requires_grad_(False)
model.eval()
nparams = sum(p.numel() for p in model.parameters()) / 1e6
print(f"[3dino] loaded {type(model).__name__} | {nparams:.0f}M params | frozen")

# quick sanity forward on one volume (uint8 -> normalized -> feature)
xb, yb = next(iter(test_loader))
with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
    f = model(preprocess_volume(xb[:1].to(device)))
print(f"[3dino] sanity: input {tuple(xb[:1].shape)} -> feature {tuple(f.shape)}")


In [ ]:
# ====== extract frozen 1024-d embeddings for train/val/test (cached on Drive) ======
@torch.no_grad()
def extract_features(loader, out_npz):
    feats, ys = [], []
    for x, y in loader:
        x = preprocess_volume(x.to(device, non_blocking=True))
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            f = model(x)
        feats.append(f.float().cpu())
        ys.append(y)
    Fx = torch.cat(feats)
    Yx = torch.cat(ys)
    np.savez(out_npz, feats=Fx.numpy(), labels=Yx.numpy())
    return Fx, Yx


FEAT_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_feats"
os.makedirs(FEAT_DIR, exist_ok=True)


def cached(split, loader):
    p = os.path.join(FEAT_DIR, f"{split}.npz")
    if os.path.exists(p):
        z = np.load(p)
        print(f"[feats] {split}: cached {z['feats'].shape}")
        return torch.from_numpy(np.array(z["feats"], copy=True)), torch.from_numpy(np.array(z["labels"], copy=True))
    Fx, Yx = extract_features(loader, p)
    print(f"[feats] {split}: extracted {tuple(Fx.shape)}")
    return Fx, Yx


Xtr, ytr = cached("Training", train_loader)
Xva, yva = cached("Validation", val_loader)
Xte, yte = cached("Test", test_loader)


In [ ]:
# ====== Linear probe on frozen 3DINO-ViT embeddings ======
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def score_row(y_true, y_pred):
    """acc / precision / recall / f1 (binary, positive = glaucoma)."""
    return dict(acc=float(accuracy_score(y_true, y_pred)),
                precision=float(precision_score(y_true, y_pred, zero_division=0)),
                recall=float(recall_score(y_true, y_pred, zero_division=0)),
                f1=float(f1_score(y_true, y_pred, zero_division=0)))


scaler = StandardScaler().fit(Xtr.numpy())
clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(scaler.transform(Xtr.numpy()), ytr.numpy())

probe = {}
for name, X, y in (("train", Xtr, ytr), ("val", Xva, yva), ("test", Xte, yte)):
    yp = clf.predict(scaler.transform(X.numpy()))
    probe[name] = score_row(y.numpy(), yp)
    print(f"[linear-probe] {name:8s} " + " | ".join(f"{k}={v:.4f}" for k, v in probe[name].items()))


In [ ]:
# ====== head finetune on frozen 3DINO-ViT (+ optional full finetune) ======
FINETUNE_ALL = False        # True: unfreeze the whole backbone (slow, more VRAM)
FINETUNE_EPOCHS = 15
BATCH = 2
GACC = 8                    # effective batch = 16
LR = 3e-4 if not FINETUNE_ALL else 1e-5
WD, PATIENCE = 0.01, 5

from monai.transforms import (Compose, RandFlip, RandRotate, RandScaleIntensity,
                              RandShiftIntensity, RandGaussianNoise)


def make_aug():
    return Compose([
        RandFlip(prob=0.5, spatial_axis=1),
        RandFlip(prob=0.5, spatial_axis=2),
        RandRotate(range_x=0.05, range_y=0.05, range_z=0.05, prob=0.4,
                   mode="bilinear", padding_mode="zeros", keep_size=True),
        RandScaleIntensity(factors=0.10, prob=0.5),
        RandShiftIntensity(offsets=10.0, prob=0.5),
        RandGaussianNoise(prob=0.3, std=5.0),
    ])


class TrainAug:
    def __init__(self, ds, tf):
        self.ds, self.tf = ds, tf
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        x, y = self.ds[i]
        x = torch.as_tensor(self.tf(x)[0])
        if x.ndim == 3:
            x = x.unsqueeze(0)
        return x, y


class DINOHead(nn.Module):
    def __init__(self, backbone, num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(1024, num_classes))
    def forward(self, x):
        x = preprocess_volume(x)
        return self.head(self.backbone(x))


net = DINOHead(model, num_classes=2)
if FINETUNE_ALL:
    for p in net.backbone.parameters():
        p.requires_grad_(True)
    print("[finetune] FULL finetune (backbone unfrozen)")
else:
    print("[finetune] head-only finetune (backbone frozen)")

train_ft = DataLoader(TrainAug(train_loader.dataset, make_aug()), batch_size=BATCH,
                      shuffle=True, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED),
                      pin_memory=(device == "cuda"))

crit = nn.CrossEntropyLoss()
params = [p for p in net.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
import math
steps_ep = math.ceil(len(train_ft) / GACC)
total = steps_ep * FINETUNE_EPOCHS
warmup = int(total * 0.05)
if warmup > 0:
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt,
        [torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warmup),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total - warmup)],
        milestones=[warmup])
else:
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
scaler = torch.amp.GradScaler("cuda",
            enabled=(USE_AMP and device == "cuda" and amp_dtype == torch.float16))


@torch.no_grad()
def eval_acc(model, loader):
    model.eval(); c = t = 0
    for x, y in loader:
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x.to(device, non_blocking=True))
        c += (logits.argmax(1) == y.to(device)).sum().item(); t += y.numel()
    return c / t


@torch.no_grad()
def eval_full(model, loader):
    """Returns acc/precision/recall/f1 over a split."""
    model.eval(); ys, ps = [], []
    for x, y in loader:
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x.to(device, non_blocking=True))
        ps.append(logits.argmax(1).cpu()); ys.append(y)
    return score_row(torch.cat(ys).numpy(), torch.cat(ps).numpy())


FT_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_ft"
os.makedirs(FT_DIR, exist_ok=True)
best_val = 0.0; bad = 0; t0 = time.time()
for ep in range(FINETUNE_EPOCHS):
    net.train(); opt.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(train_ft):
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            loss = crit(net(x), y) / GACC
        scaler.scale(loss).backward()
        if (i + 1) % GACC == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
    va = eval_acc(net, val_loader)
    print(f"ep{ep+1:02d} val={va:.4f} (best {best_val:.4f}) | {(time.time()-t0)/60:.1f} min")
    if va > best_val:
        best_val, bad = va, 0
        torch.save(net.state_dict(), os.path.join(FT_DIR, "best_3dino_head.pt"))
    else:
        bad += 1
        if bad >= PATIENCE:
            print("[early-stop] no improvement on val")
            break

net.load_state_dict(torch.load(os.path.join(FT_DIR, "best_3dino_head.pt"), map_location="cpu"))
test_metrics = eval_full(net, test_loader)
test_acc = test_metrics["acc"]
print(f"[finetune] best_val={best_val:.4f} | test: " + " ".join(f"{k}={v:.4f}" for k, v in test_metrics.items()))
print(f"[saved] best_3dino_head.pt -> {FT_DIR}")


In [ ]:
# ====== FULL finetune of 3DINO-ViT (unfreeze backbone) — resume-safe ======
# This is the main lever to push past the frozen-head 0.78. Smaller LR, more epochs.
FULL_EPOCHS = 25
FULL_LR     = 1e-5        # full finetune needs a much smaller LR than the head
FULL_WD     = 0.05
FULL_PATIENCE = 6
USE_LWD     = False       # True: layer-wise decay on backbone params (LWD^depth * lr)

FT_DIR = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_ft"
os.makedirs(FT_DIR, exist_ok=True)
FULL_CKPT = os.path.join(FT_DIR, "full_finetune.pt")
HEAD_CKPT = os.path.join(FT_DIR, "best_3dino_head.pt")

# make sure we have a DINOHead to train (defined in the head-finetune cell)
if "DINOHead" not in globals():
    class DINOHead(nn.Module):
        def __init__(self, backbone, num_classes=2, dropout=0.3):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(1024, num_classes))
        def forward(self, x):
            return self.head(preprocess_volume(x))
if "net" not in globals():
    net = DINOHead(model, num_classes=2)

# ---- start from the HEAD-finetuned checkpoint, NOT the original pretrained weights ----
if not os.path.exists(FULL_CKPT) and os.path.exists(HEAD_CKPT):
    net.load_state_dict(torch.load(HEAD_CKPT, map_location="cpu"))
    print("[full-finetune] loading head-finetuned weights (best_3dino_head.pt)")
elif not os.path.exists(FULL_CKPT):
    print("[full-finetune] WARNING: no head checkpoint found — continuing from current weights")

# re-enable gradients on the backbone
for p in net.backbone.parameters():
    p.requires_grad_(True)
print("[full-finetune] backbone unfrozen")

# optional layer-wise decay: params deeper in the network get lr * LWD^depth
def param_groups(net, base_lr, wd, lwd=0.9, use_lwd=True):
    groups = []
    decay, nolr = [], []
    named = dict(net.named_parameters())
    for name, p in named.items():
        if not p.requires_grad:
            continue
        if use_lwd and "backbone.blocks." in name:
            depth = int(name.split("backbone.blocks.")[1].split(".")[0])
            lr = base_lr * (lwd ** depth)
        else:
            lr = base_lr
        groups.append({"params": [p], "lr": lr, "weight_decay": wd})
    return groups

if USE_LWD:
    opt = torch.optim.AdamW(param_groups(net, FULL_LR, FULL_WD, lwd=LWD, use_lwd=True))
else:
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad], lr=FULL_LR, weight_decay=FULL_WD)
import math
total = math.ceil(len(train_ft) / GACC) * FULL_EPOCHS
warmup = int(total * 0.05)
if warmup > 0:
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt,
        [torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warmup),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total - warmup)],
        milestones=[warmup])
else:
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total)
scaler = torch.amp.GradScaler("cuda",
            enabled=(USE_AMP and device == "cuda" and amp_dtype == torch.float16))

# resume from an interrupted run
start_ep, full_best, bad = 0, 0.0, 0
if os.path.exists(FULL_CKPT):
    st = torch.load(FULL_CKPT, map_location="cpu")
    net.load_state_dict(st["net"]); opt.load_state_dict(st["opt"])
    scaler.load_state_dict(st["scaler"]); sched.load_state_dict(st["sched"])
    start_ep, full_best = st["epoch"], st["best_val"]
    print(f"[full-finetune] resumed at epoch {start_ep}, best_val={full_best:.4f}")

for ep in range(start_ep, FULL_EPOCHS):
    net.train(); opt.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(train_ft):
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            loss = crit(net(x), y) / GACC
        scaler.scale(loss).backward()
        if (i + 1) % GACC == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
    va = eval_acc(net, val_loader)
    print(f"ep{ep+1:02d} val={va:.4f} (best {full_best:.4f}) | {(time.time()-t0)/60:.1f} min")
    if va > full_best:
        full_best, bad = va, 0
        torch.save(net.state_dict(), os.path.join(FT_DIR, "best_3dino_full.pt"))
    else:
        bad += 1
    torch.save({"net": net.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": ep + 1, "best_val": full_best}, FULL_CKPT)
    if bad >= FULL_PATIENCE:
        print("[full-finetune] early stop (no improvement)")
        break

net.load_state_dict(torch.load(os.path.join(FT_DIR, "best_3dino_full.pt"), map_location="cpu"))
full_test_metrics = eval_full(net, test_loader)
full_test = full_test_metrics["acc"]
print(f"[full-finetune] best_val={full_best:.4f} | test: " + " ".join(f"{k}={v:.4f}" for k, v in full_test_metrics.items()))
print(f"[saved] best_3dino_full.pt + resume checkpoint -> {FT_DIR}")


In [ ]:
# ====== Fairness (per-race) + ROC/AUC + calibration on the finetuned 3DINO ======
# Harvard-GF is a fairness dataset (balanced Asian/Black/White) — race comes from the CSV.
import zipfile, csv as _csv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path as _Path
from huggingface_hub import hf_hub_download
from sklearn.metrics import (roc_auc_score, roc_curve, brier_score_loss,
                             classification_report, confusion_matrix)

# ---- rebuild the exact test-set race order used by the data build (zip namelist order) ----
race_code = {"asian": 0, "black": 1, "white": 2}
zip_path = hf_hub_download(HF_REPO, ZIP_FILE, repo_type="dataset")
csv_path = hf_hub_download(HF_REPO, CSV_FILE, repo_type="dataset")
meta = {}
with open(csv_path, newline="") as fh:
    for r in _csv.DictReader(fh):
        split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
        if split is None:
            continue
        gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
        rc = race_code.get(str(r["race"]).strip().lower(), 3)
        meta[_Path(r["filename"]).stem] = (split, gl, rc)
with zipfile.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
test_races = [meta[_Path(n).stem][2] for n in names
              if _Path(n).stem in meta and meta[_Path(n).stem][0] == "Test"]
assert len(test_races) == len(test_loader.dataset), (len(test_races), len(test_loader.dataset))
print(f"[fairness] race distribution on test:",
      {n: test_races.count(c) for n, c in (("asian", 0), ("black", 1), ("white", 2), ("other", 3))})


@torch.no_grad()
def predict_probs(net, loader):
    ps, ys = [], []
    for x, y in loader:
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = net(x.to(device, non_blocking=True))
        ps.append(torch.softmax(logits, 1)[:, 1].cpu())
        ys.append(y)
    return torch.cat(ps).numpy(), torch.cat(ys).numpy()


probs, ytest = predict_probs(net, test_loader)
pred = (probs >= 0.5).astype(int)
races = np.array(test_races)
fair_acc = (pred == ytest).mean()
fair_auc = roc_auc_score(ytest, probs)
fair_brier = brier_score_loss(ytest, probs)
fair_metrics = score_row(ytest, pred)


def sens_at_spec(y_true, probs, specs=(0.90, 0.95, 0.99)):
    """Max sensitivity achievable while keeping specificity >= each target."""
    fpr, tpr, _ = roc_curve(y_true, probs)
    out = {}
    for sp in specs:
        j = np.where((1 - fpr) >= sp)[0]
        out[f"sens@{int(sp * 100)}"] = float(tpr[j[-1]]) if len(j) else float("nan")
    return out


sens_spec = sens_at_spec(ytest, probs)
print(f"[fairness] AUC-ROC = {fair_auc:.4f}")
print("[fairness] Sensitivity @ Specificity:", sens_spec)
print("[fairness] overall " + " | ".join(f"{k}={v:.4f}" for k, v in fair_metrics.items())
      + f" | auc={fair_auc:.4f} | brier={fair_brier:.4f}")
print(classification_report(ytest, pred, target_names=["no_glaucoma", "glaucoma"]))
print("confusion matrix:\n", confusion_matrix(ytest, pred))

per_race = {}
for code, name in ((0, "asian"), (1, "black"), (2, "white"), (3, "other")):
    m = races == code
    if m.sum() == 0:
        continue
    acc = (pred[m] == ytest[m]).mean()
    auc = roc_auc_score(ytest[m], probs[m]) if len(np.unique(ytest[m])) > 1 else float("nan")
    per_race[name] = {"n": int(m.sum()), "acc": float(acc), "auc": float(auc)}
    print(f"  race={name:6s} n={m.sum():4d} acc={acc:.4f} auc={auc:.4f}")

# ---- ROC + calibration plots ----
from sklearn.calibration import calibration_curve
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fpr, tpr, _ = roc_curve(ytest, probs)
axes[0].plot(fpr, tpr, label=f"3DINO (AUC={fair_auc:.3f})")
for sp in (0.90, 0.95, 0.99):
    j = np.where((1 - fpr) >= sp)[0]
    if len(j):
        idx = j[-1]
        axes[0].scatter(fpr[idx], tpr[idx], marker="o", s=45,
                        label=f"spec={sp:.0%} sens={tpr[idx]:.2f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC (test)"); axes[0].legend()
pt, pp = calibration_curve(ytest, probs, n_bins=10)
axes[1].plot(pp, pt, marker="o", label=f"3DINO (Brier={fair_brier:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("predicted prob"); axes[1].set_ylabel("fraction positive"); axes[1].set_title("Calibration"); axes[1].legend()
fig.tight_layout()
fig.savefig(os.path.join(FT_DIR, "3dino_roc_calibration.png"), dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ====== 3DINO attention-rollout saliency (ViT method E) ======
# Roll the CLS-token attention through all 24 ViT blocks -> a 7^3 patch map,
# upsampled to the volume and overlaid on mid-slices (shortcut-learning check).
from dinov2.layers.attention import Attention
import matplotlib.pyplot as plt

test_ds = test_loader.dataset


def load_sample(ds, idx):
    x, y = ds[idx]
    return x.float().div_(255.0), int(y)      # (1,D,H,W) uint8 -> [0,1] for display


def attention_rollout(model, x):
    """Returns (D,H,W) map in [0,1]: how much the CLS token attends to each patch."""
    model.eval()
    attn_mods = [m for m in model.modules() if isinstance(m, Attention)]
    saved, hooks = {}, []
    for i, m in enumerate(attn_mods):
        name = f"a{i}"
        hooks.append(m.register_forward_pre_hook(
            lambda mod, inp, _n=name: saved.__setitem__(_n, inp[0].detach().float())))
    with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
        model(x)
    for h in hooks:
        h.remove()

    with torch.no_grad():                       # needed once the backbone is unfrozen
        N = next(iter(saved.values())).shape[1]
        rollout = torch.eye(N, device=x.device)
        for i, m in enumerate(attn_mods):
            xin = saved[f"a{i}"]
            B, Nn, C = xin.shape
            scale = (m.num_heads ** -0.5)
            qkv = m.qkv(xin).reshape(B, Nn, 3, m.num_heads, C // m.num_heads).permute(2, 0, 3, 1, 4)
            q, k = qkv[0] * scale, qkv[1]
            A = (q @ k.transpose(-2, -1)).softmax(dim=-1).mean(dim=1)[0]   # (N, N) avg over heads
            A = 0.5 * A + 0.5 * torch.eye(Nn, device=A.device)             # residual
            rollout = rollout @ A
        grid = round((rollout.shape[0] - 1) ** (1 / 3))
        cam = rollout[0, 1:].reshape(grid, grid, grid).unsqueeze(0).unsqueeze(0)
        cam = F.interpolate(cam, size=(RESOLUTION,) * 3, mode="trilinear", align_corners=False).squeeze(0, 1)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.cpu()


def show_overlay(vol, cam, title, fname, axis=0, fracs=(0.4, 0.5, 0.6)):
    fig, axes = plt.subplots(1, len(fracs), figsize=(5 * len(fracs), 5))
    for ax, fr in zip(axes, fracs):
        idx = int(vol.shape[axis] * fr)
        sl = vol[idx] if axis == 0 else vol[:, idx, :] if axis == 1 else vol[:, :, idx]
        cm = cam[idx] if axis == 0 else cam[:, idx, :] if axis == 1 else cam[:, :, idx]
        ax.imshow(sl, cmap="gray")
        ax.imshow(cm, cmap="jet", alpha=0.5, vmin=0.0, vmax=1.0)
        ax.set_title(f"sl={idx}")
        ax.axis("off")
    fig.suptitle(title)
    fig.savefig(os.path.join(FT_DIR, fname), dpi=120, bbox_inches="tight")
    plt.show()


for target_label in (0, 1):
    idx = next(i for i in range(len(test_ds)) if int(test_ds.labels[i]) == target_label)
    x, y = load_sample(test_ds, idx)
    cam = attention_rollout(model, preprocess_volume(x.unsqueeze(0)))
    print(f"sample idx={idx} true={y} | attention map {tuple(cam.shape)}")
    vol = x.squeeze(0).cpu().numpy()
    for axis, aname in ((0, "axial"), (1, "sagittal"), (2, "coronal")):
        show_overlay(vol, cam.numpy(), f"3DINO attention true={y} ({aname})",
                     f"attn_label{target_label}_{aname}.png", axis=axis)


In [ ]:
# ====== 3D Grad-CAM (ViT) on False Positives / False Negatives ======
# Clinical question: does the heatmap light up the RNFL / neuroretinal rim
# (correct signal), or is the model leaning on background / scanner artifacts?
# ViT has no conv feature maps, so Grad-CAM = CLS attention-rollout x input-gradient.
import numpy as np
import matplotlib.pyplot as plt


def attention_grid(model, x):
    """CLS attention-rollout on the patch grid (G,G,G) in [0,1]."""
    model.eval()
    attn_mods = [m for m in model.modules() if isinstance(m, Attention)]
    saved, hooks = {}, []
    for i, m in enumerate(attn_mods):
        name = f"a{i}"
        hooks.append(m.register_forward_pre_hook(
            lambda mod, inp, _n=name: saved.__setitem__(_n, inp[0].detach().float())))
    with torch.no_grad(), torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
        model(x)
    for h in hooks:
        h.remove()
    with torch.no_grad():
        N = next(iter(saved.values())).shape[1]
        rollout = torch.eye(N, device=x.device)
        for i, m in enumerate(attn_mods):
            xin = saved[f"a{i}"]
            B, Nn, C = xin.shape
            qkv = m.qkv(xin).reshape(B, Nn, 3, m.num_heads, C // m.num_heads).permute(2, 0, 3, 1, 4)
            q, k = qkv[0] * (m.num_heads ** -0.5), qkv[1]
            A = (q @ k.transpose(-2, -1)).softmax(dim=-1).mean(dim=1)[0]
            A = 0.5 * A + 0.5 * torch.eye(Nn, device=A.device)
            rollout = rollout @ A
        G = round((rollout.shape[0] - 1) ** (1 / 3))
        grid = rollout[0, 1:].reshape(G, G, G)
        grid = (grid - grid.min()) / (grid.max() - grid.min() + 1e-8)
    return grid.cpu()


def vit_gradcam(net, model, x_uint8):
    """Gradient-weighted attention (ViT Grad-CAM) -> upsampled (D,H,W) map in [0,1]."""
    xnorm = preprocess_volume(x_uint8)                     # (1,1,D,H,W) in [-1,1]
    grid = attention_grid(model, xnorm)                    # (G,G,G) rollout
    xn = xnorm.detach().clone().requires_grad_(True)
    with torch.enable_grad():
        emb = model(xn)
        logits = net.head(emb)
        cls = int(logits.argmax(1))
        grad = torch.autograd.grad(logits[0, cls], xn)[0]
        p = torch.softmax(logits.detach(), 1)[0, cls].item()
    gg = F.adaptive_avg_pool3d(grad.abs(), output_size=grid.shape).squeeze(0, 1)
    gg = (gg - gg.min()) / (gg.max() - gg.min() + 1e-8)
    cam = grid * gg                                        # gradient-weighted attention
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(RESOLUTION,) * 3,
                        mode="trilinear", align_corners=False).squeeze(0, 1)
    return cam.cpu(), cls, p


def border_mass(cam, margin=0.1):
    """Fraction of saliency within `margin` of the volume border (artifact shortcut proxy)."""
    D, H, W = cam.shape
    m = int(D * margin)
    border = torch.zeros_like(cam, dtype=torch.bool)
    border[:m] = border[-m:] = True
    border[:, :m] = border[:, -m:] = True
    border[:, :, :m] = border[:, :, -m:] = True
    return float(cam[border].sum() / (cam.sum() + 1e-8))


# reuse test predictions if the fairness cell already ran, else recompute
if "ytest" in globals() and "probs" in globals():
    yte_g, probs_g = ytest, probs
else:
    probs_g, yte_g = predict_probs(net, test_loader)
pred_g = (probs_g >= 0.5).astype(int)
fp_idx = np.where((pred_g == 1) & (yte_g == 0))[0]
fn_idx = np.where((pred_g == 0) & (yte_g == 1))[0]
print(f"[gradcam] test FP={len(fp_idx)} | FN={len(fn_idx)}")

for kind, group in (("FP", fp_idx), ("FN", fn_idx)):
    for j in group[:3]:
        xv, yv = load_sample(test_ds, int(j))
        cam, cls, p = vit_gradcam(net, model, xv.unsqueeze(0))
        bm = border_mass(cam)
        hint = ("WARNING: saliency near volume borders -> likely background/artifact shortcut"
                if bm > 0.5 else "saliency mostly central -> inspect if it lands on RNFL / neuroretinal rim")
        print(f"{kind} idx={int(j)} true={yv} pred={cls} p={p:.3f} | border_mass={bm:.2f} -> {hint}")
        vol = xv.squeeze(0).numpy()
        for axis, aname in ((0, "axial"), (1, "sagittal"), (2, "coronal")):
            show_overlay(vol, cam.numpy(), f"Grad-CAM {kind} true={yv} pred={cls} ({aname})",
                         f"gradcam_{kind}_{int(j)}_{aname}.png", axis=axis)


In [ ]:
# ====== Label-efficiency: linear probe with 1/5/10/50/100% of train labels ======
# Uses the cached frozen 3DINO embeddings — no GPU needed. Shows the pretraining
# benefit: how much accuracy survives with few labels vs. training CNNs from scratch.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

FRACTIONS = [0.01, 0.05, 0.10, 0.50, 1.00]
rng = np.random.default_rng(SEED)
Xtrn, ytrn = Xtr.numpy(), ytr.numpy()
Xvan, yvan = Xva.numpy(), yva.numpy()
Xten, yten = Xte.numpy(), yte.numpy()

eff = {"frac": [], "val": [], "test": []}
for fr in FRACTIONS:
    # stratified sampling: keep both classes in every fraction (>=1 sample per class)
    idx = []
    for c in np.unique(ytrn):
        pool = np.where(ytrn == c)[0]
        k = min(len(pool), max(1, int(round(len(pool) * fr))))
        idx.append(rng.choice(pool, k, replace=False))
    idx = np.concatenate(idx)
    sc = StandardScaler().fit(Xtrn[idx])
    clf = LogisticRegression(max_iter=2000, C=1.0)
    clf.fit(sc.transform(Xtrn[idx]), ytrn[idx])
    va = clf.score(sc.transform(Xvan), yvan)
    te = clf.score(sc.transform(Xten), yten)
    eff["frac"].append(fr); eff["val"].append(va); eff["test"].append(te)
    print(f"[label-efficiency] {fr*100:5.0f}% (n={len(idx):4d})  val={va:.4f}  test={te:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([f * 100 for f in eff["frac"]], eff["val"], marker="o", label="val")
ax.plot([f * 100 for f in eff["frac"]], eff["test"], marker="s", label="test")
ax.axhline(0.797, color="tab:red", ls="--", lw=1, label="best CNN sweep (val, 100%)")
ax.set_xlabel("% labeled train data (log scale)"); ax.set_ylabel("acc")
ax.set_xscale("log"); ax.set_ylim(0.4, 1.0); ax.grid(alpha=0.3)
ax.legend(); ax.set_title("3DINO linear-probe label efficiency")
fig.savefig(os.path.join(FT_DIR, "3dino_label_efficiency.png"), dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ====== summary + save to Drive ======
g = globals()


def _get(name, default=None):
    return g.get(name, default)


summary = {
    "method": "3DINO-ViT transfer (CC BY-NC-ND 4.0, academic use)",
    "dataset": f"Harvard-GF @ {RESOLUTION}^3",
    "backbone": "3DINO-ViT-L -> 1024-d CLS embedding",
    "linear_probe": _get("probe"),                       # {split: {acc,precision,recall,f1}}
    "head_finetune": {"best_val": _get("best_val"), "test_metrics": _get("test_metrics"),
                      "epochs": _get("FINETUNE_EPOCHS"), "finetune_all": _get("FINETUNE_ALL")},
    "full_finetune": {"best_val": _get("full_best"), "test_metrics": _get("full_test_metrics"),
                      "epochs": _get("FULL_EPOCHS"), "lr": _get("FULL_LR"),
                      "resume": os.path.exists(os.path.join(_get("FT_DIR", ""), "full_finetune.pt"))},
    "test_metrics": {"auc": _get("fair_auc"), "brier": _get("fair_brier"),
                     "sens_at_spec": _get("sens_spec"), "confusion": _get("fair_metrics")},
    "fairness_per_race": _get("per_race"),
    "label_efficiency": _get("eff"),
    "baseline_reference": {"previous_best_200_sweep": {"val": 0.7967, "test": 0.7533, "model": "enc-32-d5"}},
}
print(json.dumps(summary, indent=2))
try:
    out = "/content/drive/MyDrive/MasterBKDN/Thesis/3dino_results.json"
    with open(out, "w") as fh:
        json.dump(summary, fh, indent=2)
    print(f"[saved] 3dino_results.json -> {out}")
except Exception as e:
    print("save skipped:", e)


In [ ]:
# Done. Release the GPU immediately.
from google.colab import runtime
runtime.unassign()
